In [65]:
import pandas as pd
import os
import re

In [66]:
def projection(cols, table):
    
    fn = f'{table}.csv'
    # print(f"Getting columns {cols} from {fn}")
    
    # determine chunk size
    total_rows = sum(1 for _ in open(fn, 'r', encoding = 'utf-8')) - 1  # Minus 1 for the header
    chunk_size = max(1, total_rows // 10)  # Ensure at least 1 row per chunk to avoid division by zero

    # read specified csv file, 10% at a time
    # TODO: we are not allowed to use chunk size parameter provided by pandas
    for i, chunk in enumerate(pd.read_csv(fn, usecols = cols, chunksize = chunk_size)):
        # Write output to file for each 10%
        chunk_file = f"{table}_chunk_{i}.csv"
        chunk.to_csv(chunk_file, index = False)
        # print(f"Written chunk {i} to {chunk_file}")
        display(chunk)

In [67]:
def to_number(value):
    try:
        return int(value)
    except ValueError:
        try:
            return float(value)
        except ValueError:
            raise ValueError(f"The value '{value}' is not a number.")

In [68]:
def filtering(col, table, value, criteria):
    
    fn = f'{table}.csv'
    # print(f"Finding {table} where {col} {criteria} {value}") 
    
    # determine chunk size
    total_rows = sum(1 for _ in open(fn, 'r', encoding = 'utf-8')) - 1  # Minus 1 for the header
    chunk_size = max(1, total_rows // 10)  # Ensure at least 1 row per chunk to avoid division by zero
    
    # read specified csv file, 10% at a time
    for i, chunk in enumerate(pd.read_csv(fn, usecols = [col], chunksize = chunk_size)):
        # filter chunk file based on criteria
        if criteria == '=':
            chunk = chunk[chunk[col] == value]
        elif criteria == '>=':
            chunk = chunk[chunk[col] >= value]
        elif criteria == '<=':
            chunk = chunk[chunk[col] <= value]
        else:
            print("Invalid criteria")
            return
        if chunk.empty:
            continue
        chunk.to_csv(f"{table}_chunk_{i}.csv", index = False)
        display(chunk)
    

In [77]:
def ordering(table, col, order):
    
    fn = f'{table}.csv'
    # print(f"Ordering {table} by {col} in {order} order")
    
    # determine chunk size
    total_rows = sum(1 for _ in open(fn, 'r', encoding = 'utf-8')) - 1  # Minus 1 for the header
    chunk_size = max(1, total_rows // 10)  # Ensure at least 1 row per chunk to avoid division by zero
    
    
    # read the csv file, 10% at a time
    for i, chunk in enumerate(pd.read_csv(fn, usecols = [col], chunksize = chunk_size)):
        
        # sort the dataframe based on the column and order
        if order == 'ascending':
            chunk = chunk.sort_values(by=col, ascending=True)
        elif order == 'descending':
            chunk = chunk.sort_values(by=col, ascending=False)
        else:
            print("Invalid order")
            return
        
        chunk.to_csv(f"{table}_chunk_{i}.csv", index = False)
        display(chunk)
    

In [70]:
# Main parser function

def parse_query(query):
    # Normalize the query to lowercase for easier matching
    query = query.lower()

    if query.startswith("get"):
        # Projection
        match = re.match(r"get ([\w\, ]+) from (\w+)", query)
        if match:
            cols, table = match.groups()
            cols = cols.split(',') 
            for i in range(len(cols)):
                cols[i] = cols[i].strip()
            return projection(cols, table)
        else:
            raise ValueError("Invalid projection syntax")
    
    elif query.startswith("find"):
        # Filtering
        if "of at least" in query:
            match = re.match(r"find (\w+) that have a (\w+) of at least (\d+)", query)
            if match:
                table, col, value = match.groups()
                value = to_number(value)
                filtering(col, table, value, ">=")
            else:
                raise ValueError("Invalid filtering syntax")
        
        elif "of at most" in query:
            match = re.match(r"find (\w+) that have a (\w+) of at most (\d+)", query)
            if match:
                table, col, value = match.groups()
                value = to_number(value)
                filtering(col, table, value, "<=")
            else:
                raise ValueError("Invalid filtering syntax")
        
        elif " is " in query:
            match = re.match(r"find (\w+) that (\w+) is (.+)", query)
            if match:
                table, col, value = match.groups()
                value = to_number(value)
                filtering(col, table, value, "=")
            else:
                raise ValueError("Invalid filtering syntax")
        
        else:
            raise ValueError("Invalid filtering syntax")
        
    
    elif query.startswith("order"):
        # Ordering
        match = re.match(r"order (\w+) by (\w+) in (descending|ascending) order", query)
        if match:
            table, column, order = match.groups()
            ordering(table, column, order)
        else:
            raise ValueError("Invalid ordering syntax")
    
    else:
        raise ValueError("Unrecognized command")

- Projection: "get [col1, col2, col3, ...] from [table]"
- Filtering: "find [table] that have a [col] of at [least / most] [numeric value]", "find [table] that [col] is [value]"
- Ordering: "order [table] by [col] in [descending / ascending] order"

- ???
- Aggregation: calculate the [sum / average] of [col] for [table]
- Grouping: "group [table] by <col>"
- Inner Join: "combine rows from [table1] and [table2]  [col] have matching values"

In [78]:
# Example usage
queries = [
    # "get type, title from netflix23"
    # "get director from netflix23"
    # "find netflix23 that have a release_year of at most 2020"
    # "find netflix23 that have a release_year of at least 2020"
    # "find netflix23 that release_year is 2020"
    "order netflix23 by release_year in descending order"
    # "order netflix23 by release_year in ascending order"
]

for cmd in queries:
    try:
        print(f"query: {cmd}")
        parse_query(cmd)
        print()  # Just to have a newline for better readability
    except ValueError as e:
        print(f"Error: {e}\n")


query: order netflix23 by release_year in descending order


,release_year
1,2021
0,2020


,release_year
2,2021
3,2021


,release_year
4,2021
5,2021


,release_year
6,2021
7,1993


,release_year
8,2021
9,2021


... (3 outputs left out)

,release_year
16,2020
17,2020


,release_year
18,2021
19,2021


,release_year
20,2021
21,2018


,release_year
22,1996
